# DS2002 · Public API to Clean Table

**Lab — 2026-10-09 · Fall 2026**  

---

## Lab 07 — Public API to Clean Table

Build the weather-ingestion function you will actually reuse on the midterm. It has to pull real data, survive a flaky network, say plainly when it fell back to sample data, and hand back a table that is safe to join.

The last requirement is the one people skip. A function that returns *something* no matter what is worse than one that fails loudly, because the failure ends up in your findings instead of your console.

### Part 1 — a fetch you can trust

**TODO:** finish `get_weather`. Requirements:

- sets a timeout
- retries on 429 and 5xx with an increasing delay
- raises immediately on 400 or 404 — those are your bug, not theirs
- returns a DataFrame from the `daily` block

You wrote this shape in Wednesday's studio. Rebuild it from memory if you can.

In [2]:
import time
import pandas as pd
import requests

BASE = "https://archive-api.open-meteo.com/v1/archive"
RETRY_STATUSES = {429, 500, 502, 503, 504}


def get_weather(start, end, lat=38.03, lon=-78.51, tries=3):
  """Daily max temperature and total precipitation for a date range."""
  params = {
      "latitude": lat,
      "longitude": lon,
      "start_date": start,
      "end_date": end,
      "daily": "temperature_2m_max,precipitation_sum",
      "timezone": "America/New_York",
  }

  delay = 1
  for attempt in range(tries):
    try:
      response = requests.get(BASE, params=params, timeout=10)

      if response.status_code == 200:
        return pd.DataFrame(response.json()["daily"])

      if response.status_code in (400, 404):
        response.raise_for_status()

      if response.status_code in RETRY_STATUSES:
        if attempt == tries - 1:
          response.raise_for_status()
      else:
        response.raise_for_status()

    except requests.exceptions.RequestException as e:
      if attempt == tries - 1:
        raise e

    time.sleep(delay)
    delay *= 2

  raise RuntimeError(f"Failed to fetch weather data after {tries} attempts.")

### Part 2 — two home weekends

Pull weather for **2026-09-05** and **2026-10-03**. One range covering both is fine and is one request instead of two — be nice to free APIs.

If the call genuinely fails, use the fallback in Part 3, but you must record that you did.

In [4]:
used_fallback = False

try:
  wx = get_weather("2026-09-05", "2026-10-03")
  wx = wx[wx["time"].isin(["2026-09-05", "2026-10-03"])].reset_index(drop=True)
  source_label = "open-meteo"
except Exception as e:
  print(f"API call failed ({e}). Using fallback data.")
  used_fallback = True
  wx = FALLBACK.copy()
  source_label = "fallback sample"

### Part 3 — the fallback, labelled as a fallback

Sample values so the notebook still runs without a network. The important part is the `source` column: whoever reads your results can tell which rows are real.

In [5]:
FALLBACK = pd.DataFrame({
    "time": ["2026-09-05", "2026-10-03"],
    "temperature_2m_max": [30.1, 18.4],
    "precipitation_sum": [0.0, 22.5],
})

if used_fallback:
  wx = FALLBACK.copy()

wx["source"] = source_label

### Part 4 — make it joinable

**TODO:** produce a frame with a real datetime column named `date`, plus `temp_max`, `precip_mm`, and `source`. Rename the API's column names to yours — your pipeline should not inherit somebody else's naming.

In [6]:
wx = wx.rename(
    columns={
        "time": "date",
        "temperature_2m_max": "temp_max",
        "precipitation_sum": "precip_mm",
    }
)

wx["date"] = pd.to_datetime(wx["date"])

### Part 5 — validate before you use it

This is the habit that separates a pipeline from a script. State what must be true, and find out here rather than three cells later inside a join.

**TODO:** uncomment and make them pass.

In [7]:
assert list(wx.columns) == ["date", "temp_max", "precip_mm", "source"]
assert len(wx) == 2, f"expected 2 rows, got {len(wx)}"
assert pd.api.types.is_datetime64_any_dtype(wx["date"])
assert wx["precip_mm"].notna().all(), "missing precipitation"
assert (wx["precip_mm"] >= 0).all(), "negative rainfall is not a thing"
print("weather table is safe to join.")

weather table is safe to join.


### Part 6 — answer the question

**TODO:** which of the two dates was wetter, and by how much? Print a sentence a vendor could read, not just a number.

In [9]:
row_sep = wx[wx["date"] == "2026-09-05"].iloc[0]
row_oct = wx[wx["date"] == "2026-10-03"].iloc[0]

diff = abs(row_oct["precip_mm"] - row_sep["precip_mm"])
wetter_date = (
    "October 3, 2026"
    if row_oct["precip_mm"] > row_sep["precip_mm"]
    else "September 5, 2026"
)

print(
    f"Out of the dates, {wetter_date} was wetter by {diff:.1f} mm of"
    " precipitation."
)

Out of the dates, October 3, 2026 was wetter by 12.2 mm of precipitation.


### Part 7 — cite it

**TODO:** print the endpoint, the coordinates, the date range, when you pulled it, and whether the data is real or fallback.

In [10]:
print(f"Endpoint: {BASE}")
print(f"Coordinates: Lat 38.03, Lon -78.51")
print(f"Date Range: 2026-09-05 to 2026-10-03")
print(f"Pulled At: {pd.Timestamp.now()}")
print(f"Data Source Type: {wx['source'].iloc[0]}")

Endpoint: https://archive-api.open-meteo.com/v1/archive
Coordinates: Lat 38.03, Lon -78.51
Date Range: 2026-09-05 to 2026-10-03
Pulled At: 2026-10-10 19:38:58.018357
Data Source Type: open-meteo


### Write-up

Suppose `get_weather` returned an empty DataFrame on failure instead of raising. Walk through what happens next: you merge it onto 200 days of sales and chart rain against revenue. What does the chart show, and why would nobody catch it?

If it returned an empty DataFrame on failure instead of raising an error, a left join against 200 days of sales would produce missing weather values for those dates. Charting rain against revenue, those days would either be dropped or treated as zeros. It wouldnt be caught because the script wouldn't crash; the code would complete successfully, outputting a misleading chart.

### Stretch

Pull the same two dates for a second location — Orlando is at roughly `28.54, -81.38` — and put both cities in one long-format frame with a `city` column. That is the exact shape the midterm needs when you compare stores in different places.

In [11]:
locations = {
    "Charlottesville": {"lat": 38.03, "lon": -78.51},
    "Orlando": {"lat": 28.54, "lon": -81.38},
}

frames = []
for city, coords in locations.items():
  df = get_weather("2026-09-05", "2026-10-03", lat=coords["lat"], lon=coords["lon"])
  df = df[df["time"].isin(["2026-09-05", "2026-10-03"])].copy()
  df["city"] = city
  frames.append(df)

long_wx = pd.concat(frames, ignore_index=True)
long_wx = long_wx.rename(columns={
    "time": "date",
    "temperature_2m_max": "temp_max",
    "precipitation_sum": "precip_mm"
})
long_wx["date"] = pd.to_datetime(long_wx["date"])
long_wx["source"] = "open-meteo"

long_wx = long_wx[["city", "date", "temp_max", "precip_mm", "source"]]
print(long_wx)

              city       date  temp_max  precip_mm      source
0  Charlottesville 2026-09-05      29.4        2.5  open-meteo
1  Charlottesville 2026-10-03      22.0       14.7  open-meteo
2          Orlando 2026-09-05      32.1        3.9  open-meteo
3          Orlando 2026-10-03      30.4       14.5  open-meteo
